In [1]:
from pathlib import Path

import polars as pl
from pygments.lexer import words
from tqdm.asyncio import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

Clean-up, filters text that

- is above 200 words
- ENGLISH

In [2]:
source_dir = Path("/mnt/Fast2T/data/ai-training-set/raw/")
target_dir = Path("/mnt/Fast2T/data/ai-training-set/raw-cleaned/")


def process_file(source: Path, target: Path):
    if target.exists():
        return

    df = pl.scan_parquet(source)
    lang_col = "lang" if "lang" in df.collect_schema() else "language"

    (
        df
        .select(lang_col, "text")
        .filter(pl.col(lang_col) == "ENGLISH")
        .with_columns(words=pl.col("text").str.count_matches(r" "))
        .filter(pl.col("words") > 200)
        .select("text")
        .sink_parquet(target, compression="zstd")
    )


files = [
    (file, target_dir / directory.name / file.name)
    for directory in source_dir.iterdir()
    for file in directory.iterdir()
]

with ThreadPoolExecutor(max_workers=32) as pool:
    futures = {pool.submit(process_file, f, t): f for f, t in files}
    for future in tqdm(as_completed(futures), total=len(futures)):
        future.result()

100%|██████████| 348/348 [00:00<00:00, 905470.09it/s]


Convert document to a list of term with frequency

In [3]:
source_dir = Path("/mnt/Fast2T/data/ai-training-set/raw-cleaned/")
target_dir = Path("/mnt/Fast2T/data/ai-training-set/exploded/")

offsets = {}
cumulative = 0
for directory in sorted(source_dir.iterdir()):
    for file in sorted(directory.iterdir()):
        offsets[file] = cumulative
        cumulative += pl.scan_parquet(file).select(pl.len()).collect().item()


def process_file(source: Path, target: Path, offset: int):
    if target.exists():
        return

    target.parent.mkdir(parents=True, exist_ok=True)

    (
        pl.scan_parquet(source)
        .select("text")
        .with_row_index("id")
        .with_columns(pl.col("id") + offset)
        .select(
            "id",
            pl.col("text").str.split(" ").alias("term"),
        )
        .with_columns(
            pl.col("term").list.eval(
                pl.element().filter(
                    ~pl.element().str.contains(
                        r'[.,+*#!"§$%&/()=?<>^°\d\n\[\]{}]'
                    )
                )
            )
        )
        .with_columns(
            pl.col("term").list.len().alias("total")
        )
        .explode("term")
        .group_by("id", "term", "total")
        .len()
        .rename({"len": "count"})
        .select("id", "term", "count", "total")
        .sink_parquet(target, compression="zstd")
    )


files = [
    (file, target_dir / directory.name / file.name, offsets[file])
    for directory in source_dir.iterdir()
    for file in directory.iterdir()
]

with ThreadPoolExecutor(max_workers=4) as pool:
    futures = {pool.submit(process_file, f, t, o): f for f, t, o in files}
    for future in tqdm(as_completed(futures), total=len(futures)):
        future.result()

100%|██████████| 1846/1846 [25:35<00:00,  1.20it/s]


In [5]:
exploded_dir = Path("/mnt/Fast2T/data/ai-training-set/exploded/")
total_dir = Path("/mnt/Fast2T/data/ai-training-set/total/")

for folder in sorted(d for d in exploded_dir.iterdir() if d.is_dir()):
    target = total_dir / f"{folder.name}.parquet"
    if target.exists():
        continue

    partials = []
    corpus_total = 0
    for file in tqdm(sorted(folder.glob("*.parquet")), desc=folder.name):
        df = (
            pl.scan_parquet(file)
            .select("term", "count")
            .group_by("term")
            .agg(pl.col("count").sum())
            .collect()
        )
        corpus_total += df["count"].sum()
        partials.append(df)

    (
        pl.concat(partials).lazy()
        .group_by("term")
        .agg(pl.col("count").sum())
        .with_columns((pl.col("count") / corpus_total).alias("freq"))
        .select("term", "freq")
        .sink_parquet(target, compression="zstd", maintain_order=False)
    )

grok: 100%|██████████| 9/9 [00:27<00:00,  3.01s/it]


In [2]:
total_dir = Path("/mnt/Fast2T/data/ai-training-set/total/")

(
    pl.read_parquet(total_dir / "ai.parquet")
    .rename({"freq": "ai"})
    .join(
        pl.read_parquet(total_dir / "grok.parquet"),
        left_on="term",
        right_on="term",
        how="left",
    )
    .rename({"freq": "grokopedia"})
    .join(
        pl.read_parquet(total_dir / "human.parquet"),
        left_on="term",
        right_on="term",
        how="left",
    )
    .rename({"freq": "wikipedia"})
    .join(
        pl.read_parquet(total_dir / "web.parquet"),
        left_on="term",
        right_on="term",
        how="left",
    )
    .rename({"freq": "web 2026"})
    .join(
        pl.read_parquet(total_dir / "cc.parquet"),
        left_on="term",
        right_on="term",
        how="left",
    )
    .rename({"freq": "web 2019"})
    .drop_nulls()
    .sort("ai", descending=True)
)

term,ai,grokopedia,wikipedia,web 2026,web 2019
str,f64,f64,f64,f64,f64
"""you""",0.008902,0.000246,0.000371,0.007972,0.007187
"""use""",0.006083,0.001217,0.001555,0.003518,0.002673
"""can""",0.006062,0.000428,0.000574,0.003356,0.002309
"""your""",0.004768,0.000369,0.000147,0.005899,0.005602
"""from""",0.004301,0.008635,0.004072,0.005181,0.003637
…,…,…,…,…,…
"""wa'ad""",3.0783e-10,6.8701e-9,8.1309e-9,1.9423e-9,1.4374e-9
"""govardhana""",3.0783e-10,9.4308e-8,4.8958e-8,1.1115e-8,7.6182e-8
"""oeser""",3.0783e-10,7.8070e-9,6.9891e-8,2.2277e-8,3.4498e-8
